# Cleaning the yellow taxi data (January 2025)

## Setup

In [1]:
from pyspark.sql import SparkSession, Window, functions as F
YEAR, MONTH = 2025, 1

spark = (
    SparkSession.builder.appName("Taxi Cleaning Step By Step")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

raw = spark.read.parquet("data/yellow_tripdata_2025-01.parquet")
zones_raw = spark.read.option("header", True).option("inferSchema", True).csv("data/taxi_zone_lookup.csv")
RAW_COUNT = raw.count()
print("raw rows:", RAW_COUNT)

raw rows: 3475226


## 1. Lowercase the column names

In [2]:
df = raw.toDF(*[c.lower() for c in raw.columns])
zones = zones_raw.toDF(*[c.lower() for c in zones_raw.columns])
print(df.columns)

['vendorid', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'ratecodeid', 'store_and_fwd_flag', 'pulocationid', 'dolocationid', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee']


## 2. Add a trip_id

In [3]:
parts = [F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in df.columns]
df = df.withColumn("trip_id", F.sha2(F.concat_ws("|", *parts), 256))
df.select("trip_id", "tpep_pickup_datetime", "fare_amount").show(3, truncate=False)

+----------------------------------------------------------------+--------------------+-----------+
|trip_id                                                         |tpep_pickup_datetime|fare_amount|
+----------------------------------------------------------------+--------------------+-----------+
|a72ab80106d3b84ae55de03c0707ad3df907783b4815a46b2813df76757e25da|2025-01-01 00:18:38 |10.0       |
|3e14be4acfb6db99b7042d02ba14d56aa8f02357724508f0e818117ae7dbf7d0|2025-01-01 00:32:40 |5.1        |
|ff947d3582c20b646133484879b7dc826f3563a1684969dca56c4fbd114d4f2b|2025-01-01 00:44:04 |5.1        |
+----------------------------------------------------------------+--------------------+-----------+
only showing top 3 rows



## 3. Rule 1: missing pickup or dropoff time

In [4]:
pickup, dropoff = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")

cond_null_time = pickup.isNull() | dropoff.isNull()
print("rows breaking rule 1:", df.filter(cond_null_time).count())

rows breaking rule 1: 0


## 4. Rule 2: dropoff before pickup

In [5]:
cond_reversed = dropoff < pickup
print("rows breaking rule 2:", df.filter(cond_reversed).count())
df.filter(cond_reversed).select("tpep_pickup_datetime", "tpep_dropoff_datetime", "fare_amount").show(5, truncate=False)

rows breaking rule 2: 124
+--------------------+---------------------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|fare_amount|
+--------------------+---------------------+-----------+
|2025-01-02 12:26:00 |2025-01-02 11:29:58  |45.5       |
|2025-01-06 16:00:00 |2025-01-06 15:05:30  |24.5       |
|2025-01-15 15:00:00 |2025-01-15 14:42:48  |18.5       |
|2025-01-23 01:44:59 |2024-12-18 07:52:40  |21.9       |
|2025-01-23 12:30:00 |2025-01-23 11:44:59  |27.5       |
+--------------------+---------------------+-----------+
only showing top 5 rows



## 5. Rule 3: fare of zero or less

In [6]:
cond_bad_fare = F.col("fare_amount").isNull() | (F.col("fare_amount") <= 0)
print("rows breaking rule 3:", df.filter(cond_bad_fare).count())
df.filter(cond_bad_fare).select("tpep_pickup_datetime", "fare_amount", "total_amount", "payment_type").show(5, truncate=False)

rows breaking rule 3: 145516
+--------------------+-----------+------------+------------+
|tpep_pickup_datetime|fare_amount|total_amount|payment_type|
+--------------------+-----------+------------+------------+
|2025-01-01 00:01:41 |-7.2       |-8.54       |2           |
|2025-01-01 00:55:54 |-6.5       |-11.5       |4           |
|2025-01-01 00:56:12 |-16.3      |-21.3       |4           |
|2025-01-01 00:55:53 |-12.1      |-17.1       |2           |
|2025-01-01 00:29:35 |-7.2       |-12.2       |4           |
+--------------------+-----------+------------+------------+
only showing top 5 rows



## 6. Rule 4: pickup outside January 2025

In [7]:
from datetime import date

month_start = date(YEAR, MONTH, 1)
next_month_start = date(YEAR + (MONTH == 12), MONTH % 12 + 1, 1)
pickup_day = F.to_date(pickup)

cond_out_of_month = (pickup_day < F.lit(month_start)) | (pickup_day >= F.lit(next_month_start))
print("rows breaking rule 4:", df.filter(cond_out_of_month).count())
df.filter(cond_out_of_month).select("tpep_pickup_datetime", "tpep_dropoff_datetime").show(5, truncate=False)

rows breaking rule 4: 22
+--------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|
+--------------------+---------------------+
|2024-12-31 23:30:03 |2024-12-31 23:43:02  |
|2024-12-31 23:31:38 |2024-12-31 23:41:48  |
|2024-12-31 23:46:38 |2025-01-01 00:03:03  |
|2024-12-31 23:56:19 |2025-01-01 00:11:19  |
|2024-12-31 23:55:37 |2025-01-01 00:01:26  |
+--------------------+---------------------+
only showing top 5 rows



## 7. Rule 5: exact duplicates

In [8]:
copy_window = Window.partitionBy("trip_id").orderBy(F.lit(1))
df = df.withColumn("copy_number", F.row_number().over(copy_window))

cond_duplicate = F.col("copy_number") > 1
print("extra duplicate copies:", df.filter(cond_duplicate).count())

extra duplicate copies: 0


## 8. Combine the rules into one reject_reason

In [9]:
reason = (
    F.when(cond_null_time, "null_timestamp")
     .when(cond_reversed, "dropoff_before_pickup")
     .when(cond_bad_fare, "non_positive_fare")
     .when(cond_out_of_month, "out_of_month")
     .when(cond_duplicate, "exact_duplicate")
)
flagged = df.withColumn("reject_reason", reason).drop("copy_number")

flagged.groupBy("reject_reason").count().orderBy(F.desc("count")).show()

+--------------------+-------+
|       reject_reason|  count|
+--------------------+-------+
|                NULL|3329565|
|   non_positive_fare| 145515|
|dropoff_before_pi...|    124|
|        out_of_month|     22|
+--------------------+-------+



## 9. Split into clean and rejected, then reconcile

In [10]:
rejected = flagged.filter(F.col("reject_reason").isNotNull())
clean = flagged.filter(F.col("reject_reason").isNull()).drop("reject_reason")

clean_n, rejected_n = clean.count(), rejected.count()
print(f"raw={RAW_COUNT}  clean={clean_n}  rejected={rejected_n}")
print("reconciles:", RAW_COUNT == clean_n + rejected_n)

raw=3475226  clean=3329565  rejected=145661
reconciles: True


## 10. Enriching: Add trip_minutes, avg_mph, tip_pct and pickup_date
- `trip_minutes`: pickup to dropoff, in minutes.
- `avg_mph`: distance divided by hours, null when the trip lasted zero seconds.
- `tip_pct`: tip / fare * 100. Fare is always above zero at this point, so there is no divide-by-zero.
- `pickup_date`: the column the table will be partitioned by.

In [11]:
seconds = F.expr("timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime)")

enriched = (
    clean.withColumn("_seconds", seconds)
    .withColumn("trip_minutes", F.round(F.col("_seconds") / 60, 2))
    .withColumn("avg_mph", F.when(F.col("_seconds") > 0,
                                  F.round(F.col("trip_distance") / (F.col("_seconds") / 3600), 2)))
    .withColumn("tip_pct", F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .drop("_seconds")
)
enriched.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount",
                "tip_amount", "trip_minutes", "avg_mph", "tip_pct", "pickup_date").show(5, truncate=False)

+--------------------+---------------------+-------------+-----------+----------+------------+-------+-------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|tip_amount|trip_minutes|avg_mph|tip_pct|pickup_date|
+--------------------+---------------------+-------------+-----------+----------+------------+-------+-------+-----------+
|2025-01-31 22:55:24 |2025-01-31 23:01:37  |1.31         |7.9        |2.73      |6.22        |12.64  |34.56  |2025-01-31 |
|2025-01-14 09:21:31 |2025-01-14 09:46:15  |3.8          |20.5       |6.3       |24.73       |9.22   |30.73  |2025-01-14 |
|2025-01-28 08:59:24 |2025-01-28 09:51:39  |4.3          |39.32      |0.0       |52.25       |4.94   |0.0    |2025-01-28 |
|2025-01-18 14:52:10 |2025-01-18 15:22:58  |4.99         |30.3       |3.0       |30.8        |9.72   |9.9    |2025-01-18 |
|2025-01-09 13:51:43 |2025-01-09 14:00:16  |2.28         |12.8       |3.51      |8.55        |16.0   |27.42  |2025-01-09 |
+---------------

## 11. Enriching: Add pickup and dropoff zone names

In [12]:
pickup_zones = F.broadcast(zones.select(
    F.col("locationid").alias("pulocationid"),
    F.col("borough").alias("pickup_borough"),
    F.col("zone").alias("pickup_zone")))
dropoff_zones = F.broadcast(zones.select(
    F.col("locationid").alias("dolocationid"),
    F.col("borough").alias("dropoff_borough"),
    F.col("zone").alias("dropoff_zone")))

final = enriched.join(pickup_zones, "pulocationid", "left").join(dropoff_zones, "dolocationid", "left")

print("rows before join:", enriched.count(), "| after join:", final.count(), "(must be equal)")
final.select("pulocationid", "pickup_zone", "dolocationid", "dropoff_zone").show(5, truncate=False)

rows before join: 3329565 | after join: 3329565 (must be equal)
+------------+--------------------+------------+------------------------+
|pulocationid|pickup_zone         |dolocationid|dropoff_zone            |
+------------+--------------------+------------+------------------------+
|231         |TriBeCa/Civic Center|87          |Financial District North|
|143         |Lincoln Square West |137         |Kips Bay                |
|249         |West Village        |236         |Upper East Side North   |
|144         |Little Italy/NoLiTa |142         |Lincoln Square East     |
|209         |Seaport             |33          |Brooklyn Heights        |
+------------+--------------------+------------+------------------------+
only showing top 5 rows



# Cleansing Step: Completed